<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/16_run_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q anthropic pydantic tenacity sentence-transformers
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, os, numpy as np, pandas as pd
from tqdm import tqdm
BASE = '/content/drive/MyDrive/longevity_project'
lib = pd.read_parquet(f'{BASE}/data/processed/library.parquet')
emb = np.load(f'{BASE}/data/processed/library_embeddings.npy')
cfg = json.load(open(f'{BASE}/data/processed/library_config.json'))
shortlist = pd.read_csv(f'{BASE}/results/shortlist.csv')
scores = pd.read_parquet(f'{BASE}/results/hallmark_scores_combined.parquet')
HALLMARKS = [c[:-4] for c in scores.columns if c.endswith('_pct')]
os.makedirs(f'{BASE}/results/dossiers', exist_ok=True)

prompt_cfg = json.load(open(f'{BASE}/data/processed/review_prompt.json'))
SYSTEM, MODEL, PROMPT_VERSION = prompt_cfg['system'], prompt_cfg['model'], prompt_cfg['version']
print('Using prompt version', PROMPT_VERSION)

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(cfg['embedding_model'])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.3 MB/s eta 0:00:00
Mounted at /content/drive
Using prompt version v1


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.33k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
from pydantic import BaseModel
from typing import Literal
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

class AgingEvidence(BaseModel):
    cell: str
    animal: str
    human: str

class Dossier(BaseModel):
    drug: str
    hallmark: str
    mechanism: str
    aging_evidence: AgingEvidence
    safety: str
    confidence: Literal['High', 'Medium', 'Low']
    confidence_rationale: str
    stress_concern: bool
    key_refs: list[str]

def get_evidence(drug, hallmark, lib_, emb_, per_question=4, max_papers=10):
    rows = np.where(lib_['drug'].values == drug)[0]
    if len(rows) == 0:
        return []
    name = drug.replace('-', ' ')
    picked = []
    for template in cfg['questions'].values():
        q = embedder.encode([template.format(drug=name, hallmark=cfg['hallmark_text'][hallmark])],
                            normalize_embeddings=True)[0]
        for i in rows[np.argsort(-(emb_[rows] @ q))[:per_question]]:
            if i not in picked:
                picked.append(i)
    return [{'pmid': lib_.iloc[i]['pmid'], 'year': lib_.iloc[i]['year'], 'title': lib_.iloc[i]['title'],
             'abstract': lib_.iloc[i]['abstract'][:1500]} for i in picked[:max_papers]]

def score_context(drug, hallmark):
    s = scores.loc[drug]
    others = [cfg['hallmark_text'][h] for h in HALLMARKS if h != hallmark and s[f'{h}_pct'] >= 95]
    return (f"Screening score for {cfg['hallmark_text'][hallmark]}: {s[f'{hallmark}_pct']:.0f}th percentile of "
            f"15,542 compounds (method: {s[f'{hallmark}_source']}). Also in the top 5% for: "
            f"{', '.join(others) or 'no other hallmarks'}.")

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=20))
def review(drug, hallmark, evidence):
    text = '\n\n'.join(f"PMID {a['pmid']} ({a['year']}): {a['title']}\n{a['abstract']}" for a in evidence)
    prompt = (f"Drug: {drug}\nHallmark: {cfg['hallmark_text'][hallmark]}\n{score_context(drug, hallmark)}\n\n"
              f"Abstracts:\n{text or 'No abstracts found.'}")
    msg = client.messages.create(model=MODEL, max_tokens=8000, system=SYSTEM,
                                 messages=[{'role': 'user', 'content': prompt}])
    raw = next(b.text for b in msg.content if b.type == 'text')
    return Dossier.model_validate_json(raw[raw.find('{'): raw.rfind('}') + 1])

In [3]:
pairs = [(r.drug, hm) for r in shortlist.itertuples() for hm in r.hallmarks.split(', ')]
path_for = lambda d, hm: f'{BASE}/results/dossiers/{d}_{hm}.json'
todo = [p for p in pairs if not os.path.exists(path_for(*p))]
print(len(pairs), 'reviews in total |', len(todo), 'still to run')

evidence = {p: get_evidence(*p, lib, emb) for p in tqdm(todo)}

90 reviews in total | 90 still to run


100%|██████████| 90/90 [00:33<00:00,  2.71it/s]


In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_one(p):
    d, hm = p
    dz = review(d, hm, evidence[p]).model_dump()
    dz['supplied_pmids'] = [a['pmid'] for a in evidence[p]]
    dz['prompt_version'] = PROMPT_VERSION
    json.dump(dz, open(path_for(d, hm), 'w'), indent=2)

failed = []
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(run_one, p): p for p in todo}
    for f in tqdm(as_completed(futures), total=len(futures)):
        try:
            f.result()
        except Exception as e:
            failed.append(futures[f])
            print(futures[f], 'failed:', str(e)[:120])
print(len(failed), 'failed')

100%|██████████| 90/90 [04:27<00:00,  2.97s/it]

0 failed


In [5]:
rows = []
for d, hm in pairs:
    if not os.path.exists(path_for(d, hm)):
        continue
    dz = json.load(open(path_for(d, hm)))
    invented = [p for p in dz['key_refs'] if p not in dz['supplied_pmids']]
    rows.append({'drug': d, 'hallmark': hm, 'score_pct': scores.loc[d, f'{hm}_pct'],
                 'confidence': dz['confidence'], 'stress_concern': dz['stress_concern'],
                 'n_papers': len(dz['supplied_pmids']), 'mechanism': dz['mechanism'],
                 'evidence_cell': dz['aging_evidence']['cell'], 'evidence_animal': dz['aging_evidence']['animal'],
                 'evidence_human': dz['aging_evidence']['human'], 'safety': dz['safety'],
                 'rationale': dz['confidence_rationale'], 'key_refs': ', '.join(dz['key_refs']),
                 'invented_refs': ', '.join(invented)})

dos = pd.DataFrame(rows)
print(len(dos), 'of', len(pairs), 'dossiers collected')
bad = dos[dos['invented_refs'] != '']
print('Dossiers citing papers they were not given:', len(bad))
print(bad[['drug', 'hallmark', 'invented_refs']])

90 of 90 dossiers collected
Dossiers citing papers they were not given: 0
Empty DataFrame
Columns: [drug, hallmark, invented_refs]
Index: []


In [6]:
dos = dos[dos['invented_refs'] == '']
dos.to_parquet(f'{BASE}/results/dossiers.parquet')
dos.to_csv(f'{BASE}/results/dossiers.csv', index=False)

print(pd.crosstab(dos['hallmark'], dos['confidence']).reindex(columns=['High', 'Medium', 'Low'], fill_value=0))
print('\nFlagged for possible cell stress:', int(dos['stress_concern'].sum()), 'of', len(dos))

confidence        High  Medium  Low
hallmark                           
autophagy            0       2   13
inflammation         2       4    9
mitochondria         1       2   12
nutrient_sensing     0       2   13
proteostasis         0       1   14
senescence           1       1   13

Flagged for possible cell stress: 68 of 90


In [7]:
strong = dos[dos['confidence'].isin(['High', 'Medium']) & ~dos['stress_concern']].copy()
strong['rank'] = strong['confidence'].map({'High': 0, 'Medium': 1})
strong = strong.sort_values(['rank', 'score_pct'], ascending=[True, False])
print(strong[['drug', 'hallmark', 'confidence', 'score_pct', 'n_papers']].to_string(index=False))

multi = strong.groupby('drug')['hallmark'].apply(list)
multi = multi[multi.str.len() >= 2]
print('\nRated Medium or High on two or more hallmarks:')
for d, hms in multi.items():
    print(f'  {d}: {", ".join(hms)}')

         drug     hallmark confidence  score_pct  n_papers
  kenpaullone mitochondria       High       99.7         7
    as-605240   senescence     Medium      100.0         7
necrostatin-1 inflammation     Medium       99.8         7

Rated Medium or High on two or more hallmarks:


In [8]:
picks = [dos.sort_values('score_pct', ascending=False).query('confidence != "Low"').head(1),
         dos[dos['stress_concern']].head(1), dos[dos['confidence'] == 'Low'].sample(1, random_state=0)]
for p in picks:
    for r in p.itertuples():
        print('=' * 100, f'\n{r.drug} | {r.hallmark} | {r.confidence} | stress: {r.stress_concern}')
        print(f'MECHANISM: {r.mechanism}\nCELL: {r.evidence_cell}\nANIMAL: {r.evidence_animal}\n'
              f'HUMAN: {r.evidence_human}\nSAFETY: {r.safety}\nWHY: {r.rationale}\nREFS: {r.key_refs}')

tubacin | inflammation | Medium | stress: True
MECHANISM: Tubacin is a selective HDAC6 inhibitor. HDAC6 deacetylates alpha-tubulin (and other substrates like Hsp90, eNOS mRNA-binding complexes), and tubacin's inhibition increases tubulin/protein acetylation, altering cytoskeletal dynamics, protein homeostasis (aggresome formation), and transcript stability. These effects can modulate inflammatory signaling and endothelial function, but also disrupt proteostasis and promote apoptosis in cancer cells.
CELL: In astrocytes, tubacin blocked LPS-induced inflammatory tolerance, indicating it can prolong/enhance inflammatory responses rather than dampen them (PMID 22022450). In endothelial cell lines, tubacin increased eNOS expression and blocked proinflammatory cytokine-induced degradation of eNOS transcripts, suggesting a protective, anti-inflammatory cellular effect (PMID 31719145). In multiple myeloma and other cancer cells, tubacin caused accumulation of polyubiquitinated proteins, cell s

In [9]:
c = dos['confidence'].value_counts()
print(f"Ran the RAG review (prompt {PROMPT_VERSION}) on {len(dos)} drug-hallmark pairs across {dos['drug'].nunique()} "
      f"shortlisted drugs: {c.get('High', 0)} High, {c.get('Medium', 0)} Medium, {c.get('Low', 0)} Low confidence; "
      f"{int(dos['stress_concern'].sum())} flagged for possible cellular stress; all citations verified against "
      f"supplied abstracts. {len(strong)} pairs rated Medium or High without a stress flag"
      f"{f', and {len(multi)} drugs on two or more hallmarks' if len(multi) else ''}.")

Ran the RAG review (prompt v1) on 90 drug-hallmark pairs across 73 shortlisted drugs: 4 High, 12 Medium, 74 Low confidence; 68 flagged for possible cellular stress; all citations verified against supplied abstracts. 3 pairs rated Medium or High without a stress flag.
